<a href="https://colab.research.google.com/github/minhmax098/Brain-Tumor-CNN/blob/main/Brain_Tumor_Classfication_ViT_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive;
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install -q -r /content/drive/MyDrive/Brain_Tumor_Project/src/requirements.txt

In [3]:
import sys;
sys.path.append('/content/drive/MyDrive/Brain_Tumor_Project/src')

**1. Load Datasets**

In [4]:
import shutil
import os

source_path = '/content/drive/MyDrive/Brain_Tumor_Project/dataset'
destination_path = '/content/dataset'

# Check if the destination directory exists and copy if not
if not os.path.exists(destination_path):
    try:
        shutil.copytree(source_path, destination_path)
        print(f'Copied dataset from {source_path} to {destination_path}')
    except FileExistsError:
        print(f'Dataset already exists at {destination_path}. Skipping copy.')
    except Exception as e:
        print(f"An error occurred while copying: {e}")
else:
    print(f'Dataset already exists at {destination_path}. Skipping copy.')

from dataset import build_dataframe, print_class_distribution
df = build_dataframe()
print_class_distribution(df)
print(f"Total images: {len(df)}")

Copied dataset from /content/drive/MyDrive/Brain_Tumor_Project/dataset to /content/dataset
label           glioma  meningioma  notumor  pituitary
original_split                                        
Testing            400         400      400        400
Training          1400        1400     1400       1400
Total images: 7200


**2. Check Augmentation**

In [5]:
from preprocessing import get_train_transforms
import numpy as np
dummy = (np.random.rand(256,256,3)*255).astype('uint8')
out = get_train_transforms()(image=dummy)["image"]
print(out.shape)

torch.Size([3, 224, 224])


/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:33: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),


**3. Definite model om GPU**

In [6]:
from model import HybridCNNTransformer
import torch
model = HybridCNNTransformer(pretrained_backbone=True).to("cuda")
dummy = torch.randn(2,3,224,224).to("cuda")
logits = model(dummy)
print(logits.shape)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 223MB/s]
/content/drive/MyDrive/Brain_Tumor_Project/src/transformer_head.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


torch.Size([2, 4])


**4. Test training loop**

In [7]:
from train import run_two_stage_training
from torch.utils.data import TensorDataset, DataLoader
import torch
x = torch.randn(16,3,224,224);
y = torch.randint(0,4,(16,))
loader = DataLoader(TensorDataset(x,y), batch_size=4)
history = run_two_stage_training(model, loader, loader)

[Stage 1] trainable params: {'total': 37195844, 'trainable': 13687812, 'frozen': 23508032}
[S1 E1/1] train_acc=0.3125 val_acc=0.3125
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/1] train_acc=0.3125 val_acc=0.3125


**5. Try DataLoader with 200 images**

In [8]:
from preprocessing import BrainTumorDataset, get_train_transforms
small_df = df.sample(200, random_state=42).reset_index(drop=True)
ds = BrainTumorDataset(small_df, get_train_transforms())
loader = DataLoader(ds, batch_size=8, shuffle=True)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape)

/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:33: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),


torch.Size([8, 3, 224, 224]) torch.Size([8])


**6. Try Train with 1-2 epoch on real data (small subset)**

In [9]:
model2 = HybridCNNTransformer(pretrained_backbone=True).to("cuda")
history = run_two_stage_training(model2, loader, loader)

[Stage 1] trainable params: {'total': 37195844, 'trainable': 13687812, 'frozen': 23508032}
[S1 E1/1] train_acc=0.3050 val_acc=0.2550
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/1] train_acc=0.2550 val_acc=0.3550


In [10]:
from torch.utils.data import DataLoader
import time
full_ds = BrainTumorDataset(df, get_train_transforms())
full_loader = DataLoader(full_ds, batch_size=32, shuffle=True, num_workers=2)
t0 = time.time()
for xb, yb in full_loader: pass
print(f"1 epoch scans through lost data: {time.time()-t0:.1f} s")

1 epoch scans through lost data: 20.4 s


In [11]:
from dataset import build_dataframe
df = build_dataframe()
# build lại vì DATA_ROOT đổi
from preprocessing import BrainTumorDataset, get_train_transforms
from torch.utils.data import DataLoader
import time
full_ds = BrainTumorDataset(df, get_train_transforms())
full_loader = DataLoader(full_ds, batch_size=32, shuffle=True, num_workers=4)
t0 = time.time()
for xb, yb in full_loader:
    pass
print(f"1 epoch (local disk): {time.time()-t0:.1f}s")

1 epoch (local disk): 10.7s


In [12]:
import os
from pathlib import Path
import sys
import importlib

config_path = Path('/content/drive/MyDrive/Brain_Tumor_Project/src/config.py')

# Default values for the missing configurations
config_content_to_add = """
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
K_FOLDS = 5
RESULTS_DIR = '/content/results' # Colab local path
CHECKPOINT_DIR = '/content/checkpoints' # Colab local path
STAGE1_EPOCHS = 20
STAGE2_EPOCHS = 40
NUM_LAYERS = 2
EMBED_DIM = 256
"""

# Check if config.py exists and if the variables are defined
if not config_path.exists():
    # Ensure the directory for config.py exists
    config_path.parent.mkdir(parents=True, exist_ok=True)
    # If the file doesn't exist, create it with the content
    with open(config_path, 'w') as f:
        f.write(config_content_to_add.strip())
    print(f"Created config.py and added default configurations at {config_path}")
elif config_path.is_file(): # Ensure it's a file, not a directory
    # If the file exists, check if the variables are present.
    with open(config_path, 'r') as f:
        current_content = f.read()

    missing_variables = []
    for var_name in ["DEVICE", "BATCH_SIZE", "K_FOLDS", "RESULTS_DIR", "CHECKPOINT_DIR", "STAGE1_EPOCHS", "STAGE2_EPOCHS", "NUM_LAYERS", "EMBED_DIM"]:
        # A simple check if the variable name is in the file content
        if not (f' {var_name} =' in current_content or f'{var_name}=' in current_content):
            missing_variables.append(var_name)

    if missing_variables:
        print(f"Adding missing configurations to {config_path}: {', '.join(missing_variables)}")
        with open(config_path, 'a') as f: # Append mode
            f.write(config_content_to_add)
    else:
        print(f"All required configurations already present in {config_path}.")
else:
    print(f"Warning: {config_path} exists but is not a regular file. Skipping configuration update.")

# --- FIX: Remove modules from sys.modules to force a fresh import ---
if 'config' in sys.modules:
    print("Removing 'config' from sys.modules to force a reload.")
    del sys.modules['config']
if 'run_full_experiment' in sys.modules:
    print("Removing 'run_full_experiment' from sys.modules to force a reload.")
    del sys.modules['run_full_experiment']

# Now attempt to run the original code
from run_full_experiment import run_full_experiment
results = run_full_experiment()

Adding missing configurations to /content/drive/MyDrive/Brain_Tumor_Project/src/config.py: DEVICE, BATCH_SIZE, K_FOLDS, RESULTS_DIR, CHECKPOINT_DIR, STAGE1_EPOCHS, STAGE2_EPOCHS
Removing 'config' from sys.modules to force a reload.
Không có tiến độ cũ, bắt đầu từ fold 1.

===== FOLD 1/5 =====
-- Training CNN-only baseline --


/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:33: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),
/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:48: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),


[Stage 1] trainable params: {'total': 23516228, 'trainable': 8196, 'frozen': 23508032}
[S1 E1/5] train_acc=0.7552 val_acc=0.8257
[S1 E2/5] train_acc=0.8311 val_acc=0.8694
[S1 E3/5] train_acc=0.8500 val_acc=0.8694
[S1 E4/5] train_acc=0.8611 val_acc=0.8701
[S1 E5/5] train_acc=0.8644 val_acc=0.9014
[Stage 2] trainable params: {'total': 23516228, 'trainable': 14972932, 'frozen': 8543296}
[S2 E1/10] train_acc=0.8792 val_acc=0.9069
[S2 E2/10] train_acc=0.8899 val_acc=0.9132
[S2 E3/10] train_acc=0.9021 val_acc=0.9181
[S2 E4/10] train_acc=0.9094 val_acc=0.9354
[S2 E5/10] train_acc=0.9160 val_acc=0.9375
[S2 E6/10] train_acc=0.9224 val_acc=0.9354
[S2 E7/10] train_acc=0.9259 val_acc=0.9361
[S2 E8/10] train_acc=0.9229 val_acc=0.9403
[S2 E9/10] train_acc=0.9290 val_acc=0.9403
[S2 E10/10] train_acc=0.9304 val_acc=0.9389
[Fold 1] CNN-only: {'accuracy': 0.9388888888888889, 'precision_macro': 0.9393773335319676, 'recall_macro': 0.938888888888889, 'f1_macro': 0.9390008437316508, 'auc_macro': np.float64(

/content/drive/MyDrive/Brain_Tumor_Project/src/transformer_head.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[S1 E1/5] train_acc=0.5033 val_acc=0.7132
[S1 E2/5] train_acc=0.7781 val_acc=0.8194
[S1 E3/5] train_acc=0.7865 val_acc=0.7681
[S1 E4/5] train_acc=0.7345 val_acc=0.7875
[S1 E5/5] train_acc=0.7644 val_acc=0.7833
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/10] train_acc=0.7925 val_acc=0.8222
[S2 E2/10] train_acc=0.8063 val_acc=0.8181
[S2 E3/10] train_acc=0.8128 val_acc=0.8153
[S2 E4/10] train_acc=0.8036 val_acc=0.8174
[S2 E5/10] train_acc=0.8163 val_acc=0.8361
[S2 E6/10] train_acc=0.8099 val_acc=0.8361
[S2 E7/10] train_acc=0.8194 val_acc=0.8375
[S2 E8/10] train_acc=0.8196 val_acc=0.8271
[S2 E9/10] train_acc=0.8113 val_acc=0.8465
[S2 E10/10] train_acc=0.8189 val_acc=0.8396
[Fold 1] CNN+Transformer: {'accuracy': 0.8395833333333333, 'precision_macro': 0.8368653307343461, 'recall_macro': 0.8395833333333333, 'f1_macro': 0.8361443728735829, 'auc_macro': np.float64(0.9634104938271605)}
-- Fitting PCA-SVM baseline (ablation) --
[Fold 1] PCA-SVM

...

[S1 E1/5] train_acc=0.4830 val_acc=0.6896
[S1 E2/5] train_acc=0.7576 val_acc=0.8028
[S1 E3/5] train_acc=0.7953 val_acc=0.8028
[S1 E4/5] train_acc=0.7658 val_acc=0.7486
[S1 E5/5] train_acc=0.7828 val_acc=0.7854
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/10] train_acc=0.8038 val_acc=0.8056
[S2 E2/10] train_acc=0.8111 val_acc=0.8132
[S2 E3/10] train_acc=0.8151 val_acc=0.8333
[S2 E4/10] train_acc=0.8108 val_acc=0.8049
[S2 E5/10] train_acc=0.8161 val_acc=0.8104
[S2 E6/10] train_acc=0.8184 val_acc=0.8132
[S2 E7/10] train_acc=0.8082 val_acc=0.8125
[S2 E8/10] train_acc=0.8106 val_acc=0.8069
[S2 E9/10] train_acc=0.8179 val_acc=0.8299
[S2 E10/10] train_acc=0.8163 val_acc=0.7986
[Fold 4] CNN+Transformer: {'accuracy': 0.7986111111111112, 'precision_macro': 0.797827758134729, 'recall_macro': 0.798611111111111, 'f1_macro': 0.793900027810233, 'auc_macro': np.float64(0.9512011316872429)}
-- Fitting PCA-SVM baseline (ablation) --
[Fold 4] PCA-SVM: {

/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:33: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),
/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:48: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),


[Stage 1] trainable params: {'total': 23516228, 'trainable': 8196, 'frozen': 23508032}
[S1 E1/5] train_acc=0.7566 val_acc=0.8514
[S1 E2/5] train_acc=0.8281 val_acc=0.8528
[S1 E3/5] train_acc=0.8455 val_acc=0.8819
[S1 E4/5] train_acc=0.8573 val_acc=0.8854
[S1 E5/5] train_acc=0.8646 val_acc=0.8819
[Stage 2] trainable params: {'total': 23516228, 'trainable': 14972932, 'frozen': 8543296}
[S2 E1/10] train_acc=0.8778 val_acc=0.9083
[S2 E2/10] train_acc=0.8891 val_acc=0.9146
[S2 E3/10] train_acc=0.9002 val_acc=0.9194
[S2 E4/10] train_acc=0.9118 val_acc=0.9250
[S2 E5/10] train_acc=0.9179 val_acc=0.9243
[S2 E6/10] train_acc=0.9200 val_acc=0.9354
[S2 E7/10] train_acc=0.9212 val_acc=0.9306
[S2 E8/10] train_acc=0.9269 val_acc=0.9292
[S2 E9/10] train_acc=0.9264 val_acc=0.9264
[S2 E10/10] train_acc=0.9273 val_acc=0.9333
[Fold 5] CNN-only: {'accuracy': 0.9333333333333333, 'precision_macro': 0.9330922937387169, 'recall_macro': 0.9333333333333333, 'f1_macro': 0.9331555848150076, 'auc_macro': np.float64

/content/drive/MyDrive/Brain_Tumor_Project/src/transformer_head.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[S1 E1/5] train_acc=0.4741 val_acc=0.7896
[S1 E2/5] train_acc=0.7641 val_acc=0.8292
[S1 E3/5] train_acc=0.7905 val_acc=0.8181
[S1 E4/5] train_acc=0.7694 val_acc=0.8264
[S1 E5/5] train_acc=0.7804 val_acc=0.7812
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/10] train_acc=0.7649 val_acc=0.7951
[S2 E2/10] train_acc=0.7778 val_acc=0.8181
[S2 E3/10] train_acc=0.7774 val_acc=0.8097
[S2 E4/10] train_acc=0.7932 val_acc=0.8028
[S2 E5/10] train_acc=0.7962 val_acc=0.8153
[S2 E6/10] train_acc=0.8068 val_acc=0.8236
[S2 E7/10] train_acc=0.8030 val_acc=0.8194
[S2 E8/10] train_acc=0.7983 val_acc=0.8174
[S2 E9/10] train_acc=0.8003 val_acc=0.8167
[S2 E10/10] train_acc=0.8087 val_acc=0.8076
[Fold 5] CNN+Transformer: {'accuracy': 0.8076388888888889, 'precision_macro': 0.8059417452169767, 'recall_macro': 0.807638888888889, 'f1_macro': 0.802966116370887, 'auc_macro': np.float64(0.9517367541152264)}
-- Fitting PCA-SVM baseline (ablation) --
[Fold 5] PCA-SVM: 

In [13]:
 !cat /content/drive/MyDrive/Brain_Tumor_Project/src/pca_svm_baseline.py | grep probability

    pipe = Pipeline([("scaler", StandardScaler()), ("pca", PCA()), ("svm", SVC(kernel="rbf", probability=True))])


In [14]:
from train import run_two_stage_training
from model import HybridCNNTransformer
from torch.utils.data import TensorDataset, DataLoader
import torch, os
x = torch.randn(16,3,224,224);
y = torch.randint(0,4,(16,))
loader = DataLoader(TensorDataset(x,y), batch_size=4)
model_test = HybridCNNTransformer(pretrained_backbone=False).to("cuda")
history = run_two_stage_training(model_test, loader, loader)
print(os.listdir("/content/drive/MyDrive/Brain_Tumor_Project/models"))

/content/drive/MyDrive/Brain_Tumor_Project/src/transformer_head.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[Stage 1] trainable params: {'total': 37195844, 'trainable': 13687812, 'frozen': 23508032}
[S1 E1/1] train_acc=0.1875 val_acc=0.1875
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/1] train_acc=0.1875 val_acc=0.1875
['best_mobilenetv2_stage1.pth', 'best_mobilenetv2_finetuned_loss.pth', 'best_mobilenetv2_finetuned_acc.pth', 'best_resnet50_stage1.pth', 'best_resnet50_finetuned_loss.pth', 'best_resnet50_finetuned_acc.pth', 'model_best.pt', 'model_latest.pt', 'cnn_only_best.pt', 'cnn_only_latest.pt', 'hybrid_latest.pt', 'hybrid_best.pt']


In [15]:
from dataset import build_dataframe, get_holdout_split
from preprocessing import BrainTumorDataset, get_train_transforms, get_eval_transforms
from torch.utils.data import DataLoader
import time
df = build_dataframe()
train_df, val_df, test_df = get_holdout_split(df)
train_loader = DataLoader(BrainTumorDataset(train_df, get_train_transforms()), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(BrainTumorDataset(val_df, get_eval_transforms()), batch_size=32, num_workers=2)
model = HybridCNNTransformer(pretrained_backbone=True).to("cuda")
t0=time.time()
history = run_two_stage_training(model, train_loader, val_loader)
print(f"Tổng thời gian 4 epoch (2+2): {time.time()-t0:.1f}s")

/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:33: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),
/content/drive/MyDrive/Brain_Tumor_Project/src/preprocessing.py:48: UserWarning: Using lambda is incompatible with multiprocessing. Consider using regular functions or partial().
  A.Lambda(image=lambda img, **kw: apply_clahe(img)),


[Stage 1] trainable params: {'total': 37195844, 'trainable': 13687812, 'frozen': 23508032}


/content/drive/MyDrive/Brain_Tumor_Project/src/transformer_head.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


[S1 E1/2] train_acc=0.4566 val_acc=0.7764
[S1 E2/2] train_acc=0.7658 val_acc=0.8083
[Stage 2] trainable params: {'total': 37195844, 'trainable': 28652548, 'frozen': 8543296}
[S2 E1/2] train_acc=0.8146 val_acc=0.8264
[S2 E2/2] train_acc=0.8191 val_acc=0.8250
Tổng thời gian 4 epoch (2+2): 90.7s


In [16]:
import importlib
import train
importlib.reload(train)
from train import run_two_stage_training

In [17]:
config_file_path = '/content/drive/MyDrive/Brain_Tumor_Project/src/config.py'
try:
    with open(config_file_path, 'r') as f:
        content = f.read()
        if 'CHECKPOINT_DIR' in content:
            print(f"'CHECKPOINT_DIR' is defined in {config_file_path}")
        else:
            print(f"'CHECKPOINT_DIR' is NOT defined in {config_file_path}")
except FileNotFoundError:
    print(f"Error: The file {config_file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

'CHECKPOINT_DIR' is defined in /content/drive/MyDrive/Brain_Tumor_Project/src/config.py
